# 01 — Gerar população de agentes sintéticos em CSV

Este notebook cria a base populacional de agentes para o projeto **Synthetic Audiences ABM**.

Ele gera:

1. `agents_population.csv` — base completa dos agentes.
2. `agents_population_ptbr_semicolon.csv` — mesma base com separador `;`, melhor para Excel/Sheets em português.
3. `agents_population_sample.csv` — amostra para inspeção rápida.
4. `agents_population_distribution_audit.csv` — auditoria das distribuições observadas.
5. `agents_population_target_comparison.csv` — comparação entre distribuição observada e pesos esperados, quando houver alvo no YAML.

> Rode este notebook a partir da raiz do repositório `synthetic-audiences-abm`.

## 1. Configuração inicial

Se estiver no Google Colab, primeiro clone o repositório e instale o pacote:

```python
# !git clone <URL_DO_SEU_REPOSITORIO>
# %cd synthetic-audiences-abm
# !pip install -e .
```

Em ambiente local, abra o notebook dentro da pasta do projeto e siga as células abaixo.

In [ ]:
from pathlib import Path
import sys
import subprocess
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)


def find_project_root(start: Path) -> Path:
    """Find repo root by looking for pyproject.toml and src/synthetic_audiences."""
    start = start.resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "synthetic_audiences").exists():
            return candidate

    # Fallback útil quando o notebook foi gerado/baixado no ambiente do ChatGPT.
    fallback = Path("/mnt/data/synthetic-audiences-abm")
    if fallback.exists():
        return fallback

    raise FileNotFoundError(
        "Não encontrei a raiz do projeto. Rode o notebook dentro do repositório synthetic-audiences-abm."
    )


PROJECT_DIR = find_project_root(Path.cwd())
SRC_DIR = PROJECT_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("PROJECT_DIR:", PROJECT_DIR)
print("SRC_DIR:", SRC_DIR)

## 2. Importar funções do projeto

Esta célula tenta importar as funções do pacote. Se ainda não estiver instalado, instala automaticamente o repositório em modo editável.

In [ ]:
try:
    from synthetic_audiences.population import generate_profiles
    from synthetic_audiences.config import DEFAULT_CONFIG_PATH, load_config
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(PROJECT_DIR)])
    from synthetic_audiences.population import generate_profiles
    from synthetic_audiences.config import DEFAULT_CONFIG_PATH, load_config

config = load_config(DEFAULT_CONFIG_PATH)
print("Config carregado de:", DEFAULT_CONFIG_PATH)
print("Projeto:", config.get("project", {}).get("name"))

## 3. Definir parâmetros da população

Ajuste `N_AGENTS` conforme o tamanho desejado da base.

Sugestão inicial:

- `300` para demo rápida.
- `1_000` para apresentação/protótipo.
- `5_000` ou mais para simulações mais estáveis.

In [ ]:
N_AGENTS = 1_000
SEED = 42
POPULATION_NAME = f"agents_population_n{N_AGENTS}_seed{SEED}"

OUTPUT_DIR = PROJECT_DIR / "data" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AGENTS_CSV = OUTPUT_DIR / f"{POPULATION_NAME}.csv"
AGENTS_CSV_PTBR = OUTPUT_DIR / f"{POPULATION_NAME}_ptbr_semicolon.csv"
SAMPLE_CSV = OUTPUT_DIR / f"{POPULATION_NAME}_sample.csv"
AUDIT_CSV = OUTPUT_DIR / f"{POPULATION_NAME}_distribution_audit.csv"
TARGET_COMPARISON_CSV = OUTPUT_DIR / f"{POPULATION_NAME}_target_comparison.csv"

print("Base principal:", AGENTS_CSV)
print("Base PT-BR ;:", AGENTS_CSV_PTBR)
print("Amostra:", SAMPLE_CSV)
print("Auditoria:", AUDIT_CSV)
print("Comparação com alvos:", TARGET_COMPARISON_CSV)

## 4. Gerar a base de agentes

A função `generate_profiles` aplica os pesos populacionais e gera variáveis demográficas, survey-based e persona-based.

In [ ]:
df_agents = generate_profiles(
    n=N_AGENTS,
    output_csv=AGENTS_CSV,
    config_path=DEFAULT_CONFIG_PATH,
    seed=SEED,
)

# Versão PT-BR: separador ; e encoding utf-8-sig para abrir bem em Excel/Sheets.
df_agents.to_csv(AGENTS_CSV_PTBR, index=False, sep=";", encoding="utf-8-sig")

# Amostra para inspeção manual.
df_agents.sample(min(50, len(df_agents)), random_state=SEED).to_csv(
    SAMPLE_CSV,
    index=False,
    sep=";",
    encoding="utf-8-sig",
)

print(f"OK — {len(df_agents):,} agentes gerados".replace(",", "."))
print("CSV principal salvo em:", AGENTS_CSV)
print("CSV PT-BR salvo em:", AGENTS_CSV_PTBR)
print("Amostra salva em:", SAMPLE_CSV)

df_agents.head(5)

## 5. Validação básica da base

Esta etapa checa:

- número de linhas;
- IDs únicos;
- colunas principais;
- valores nulos;
- prévia das personas.

In [ ]:
required_columns = [
    "id_persona",
    "idade",
    "gênero",
    "raça/cor",
    "renda familiar mensal",
    "renda_pc",
    "identificação política",
    "ideologia",
    "geração",
    "segmento_e_bolhas",
    "tipo_de_território",
    "uf",
    "tipo_de_fontes_de_informação",
    "Religiosidade",
    "Família",
    "Honestidade",
    "Meritocracia",
    "Ideologia",
    "Discriminação",
    "Sexualidade",
    "Comportamento_social",
    "Identidade_nacional",
    "Insegurança",
    "Confiança",
    "Persona",
]

missing_columns = [c for c in required_columns if c not in df_agents.columns]

validation = {
    "n_rows": len(df_agents),
    "n_columns": len(df_agents.columns),
    "id_persona_unique": df_agents["id_persona"].is_unique,
    "missing_required_columns": missing_columns,
    "total_null_cells": int(df_agents.isna().sum().sum()),
}

validation

In [ ]:
df_agents[[
    "id_persona",
    "idade",
    "gênero",
    "raça/cor",
    "renda familiar mensal",
    "geração",
    "segmento_e_bolhas",
    "tipo_de_território",
    "uf",
    "identificação política",
    "ideologia",
    "Persona",
]].head(3)

## 6. Auditoria das distribuições observadas

Esta célula gera uma tabela longa com `n` e `%` por variável/categoria.

In [ ]:
def distribution_table(df: pd.DataFrame, column: str) -> pd.DataFrame:
    out = (
        df[column]
        .value_counts(dropna=False)
        .rename_axis("categoria")
        .reset_index(name="n")
    )
    out.insert(0, "variavel", column)
    out["percentual_observado"] = out["n"] / len(df)
    out["percentual_observado_fmt"] = (out["percentual_observado"] * 100).round(2).astype(str) + "%"
    return out


audit_columns = [
    "segmento_e_bolhas",
    "geração",
    "idade",
    "gênero",
    "raça/cor",
    "renda_pc",
    "renda familiar mensal",
    "identificação política",
    "ideologia",
    "tipo_de_território",
    "uf",
    "tipo_de_fontes_de_informação",
]

df_audit = pd.concat(
    [distribution_table(df_agents, col) for col in audit_columns if col in df_agents.columns],
    ignore_index=True,
)

df_audit.to_csv(AUDIT_CSV, index=False, sep=";", encoding="utf-8-sig")

print("Auditoria salva em:", AUDIT_CSV)
df_audit.head(20)

## 7. Comparar distribuição observada com pesos-alvo do YAML

A comparação direta é feita para variáveis que têm distribuição-base explícita no arquivo de configuração.

> Observação: quanto menor o `N_AGENTS`, maior a variação aleatória esperada entre o alvo e o observado.

In [ ]:
def target_comparison(df: pd.DataFrame, column: str, target_weights: dict) -> pd.DataFrame:
    observed = df[column].value_counts(normalize=True, dropna=False).to_dict()
    categories = sorted(set(target_weights.keys()) | set(observed.keys()))

    rows = []
    for cat in categories:
        target = float(target_weights.get(cat, 0.0))
        obs = float(observed.get(cat, 0.0))
        rows.append({
            "variavel": column,
            "categoria": cat,
            "percentual_alvo": target,
            "percentual_observado": obs,
            "diferenca_pontos_percentuais": (obs - target) * 100,
        })
    return pd.DataFrame(rows)


target_tables = []

# Segmentos e bolhas: pesos populacionais por segmento.
segment_targets = {
    segment: values["population"]
    for segment, values in config["segment_distributions"].items()
}
target_tables.append(target_comparison(df_agents, "segmento_e_bolhas", segment_targets))

# Distribuições-base nacionais explícitas no YAML.
base = config.get("base_distributions", {})
for column, weights in [
    ("geração", base.get("geração", {})),
    ("gênero", base.get("gênero", {})),
    ("raça/cor", base.get("raça/cor", {})),
    ("renda_pc", base.get("renda_pc", {})),
]:
    if weights and column in df_agents.columns:
        target_tables.append(target_comparison(df_agents, column, weights))

df_target_comparison = pd.concat(target_tables, ignore_index=True)
df_target_comparison["percentual_alvo_fmt"] = (df_target_comparison["percentual_alvo"] * 100).round(2).astype(str) + "%"
df_target_comparison["percentual_observado_fmt"] = (df_target_comparison["percentual_observado"] * 100).round(2).astype(str) + "%"
df_target_comparison["diferenca_pontos_percentuais"] = df_target_comparison["diferenca_pontos_percentuais"].round(2)

df_target_comparison.to_csv(TARGET_COMPARISON_CSV, index=False, sep=";", encoding="utf-8-sig")

print("Comparação com alvos salva em:", TARGET_COMPARISON_CSV)
df_target_comparison.sort_values(["variavel", "categoria"]).head(40)

## 8. Visão rápida por segmento

Esta célula ajuda a ver se a população gerada faz sentido por grupo psicográfico/político-cultural.

In [ ]:
segment_summary = (
    df_agents
    .groupby("segmento_e_bolhas", as_index=False)
    .agg(
        n=("id_persona", "count"),
        pct=("id_persona", lambda s: len(s) / len(df_agents)),
        top_geração=("geração", lambda s: s.value_counts().index[0]),
        top_uf=("uf", lambda s: s.value_counts().index[0]),
        top_território=("tipo_de_território", lambda s: s.value_counts().index[0]),
        top_politica=("identificação política", lambda s: s.value_counts().index[0]),
        top_ideologia=("ideologia", lambda s: s.value_counts().index[0]),
    )
    .sort_values("n", ascending=False)
)

segment_summary["pct_fmt"] = (segment_summary["pct"] * 100).round(2).astype(str) + "%"
segment_summary

## 9. Gerar múltiplas populações de uma vez — opcional

Use esta célula se quiser gerar arquivos diferentes para demo, piloto e simulação maior.

Por padrão, ela está configurada com tamanhos razoáveis. Ajuste conforme o custo/tempo desejado.

In [ ]:
RUN_MULTIPLE_SCENARIOS = False

scenarios = [
    {"name": "demo_300", "n": 300, "seed": 42},
    {"name": "pilot_1000", "n": 1_000, "seed": 42},
    {"name": "simulation_5000", "n": 5_000, "seed": 42},
]

if RUN_MULTIPLE_SCENARIOS:
    generated_files = []
    for scenario in scenarios:
        name = scenario["name"]
        n = scenario["n"]
        seed = scenario["seed"]

        csv_path = OUTPUT_DIR / f"agents_population_{name}.csv"
        csv_path_ptbr = OUTPUT_DIR / f"agents_population_{name}_ptbr_semicolon.csv"

        df_scenario = generate_profiles(
            n=n,
            output_csv=csv_path,
            config_path=DEFAULT_CONFIG_PATH,
            seed=seed,
        )
        df_scenario.to_csv(csv_path_ptbr, index=False, sep=";", encoding="utf-8-sig")
        generated_files.append({"scenario": name, "n": n, "csv": str(csv_path), "csv_ptbr": str(csv_path_ptbr)})

    pd.DataFrame(generated_files)
else:
    print("RUN_MULTIPLE_SCENARIOS está False. Altere para True para gerar múltiplas bases.")

## 10. Checklist final

Depois de rodar o notebook, confira se os arquivos foram criados em `data/output/`.

In [ ]:
created_files = [
    AGENTS_CSV,
    AGENTS_CSV_PTBR,
    SAMPLE_CSV,
    AUDIT_CSV,
    TARGET_COMPARISON_CSV,
]

for file in created_files:
    print("OK" if file.exists() else "FALTANDO", "—", file.relative_to(PROJECT_DIR) if file.exists() else file)